# Discovery, Validation, And Artifact Inspection

This notebook runs discovery and validation, then loads the richer cluster summary outputs so you can inspect motifs without reading raw JSON by hand.

If the training run used `dataset_selection`, discovery and validation will automatically use the same derived selected split manifest when you point them at the same experiment config.

In [ ]:
from google.colab import drive
from pathlib import Path
import json
import pandas as pd

drive.mount('/content/drive')
repo_root = Path('/content/predictive-circuit-coding')
checkpoint_dir = repo_root / 'artifacts' / 'checkpoints'

In [ ]:
if not repo_root.exists():
    !git clone https://github.com/jacobposchl/predictive-circuit-coding.git {repo_root}
%cd {repo_root}
!pip install -e .[notebook]

In [ ]:
import ipywidgets as widgets
from predictive_circuit_coding.utils import NotebookStageReporter, verify_paths_exist

reporter = NotebookStageReporter(name='colab-discover', expected_duration='1-5 minutes for debug runs, longer for larger discovery passes')
reporter.banner('Discovery And Validation', subtitle='Frozen-token discovery, validation controls, and report inspection')
experiment_config = repo_root / 'configs' / 'pcc' / 'predictive_circuit_coding_base.yaml'
data_config = repo_root / 'configs' / 'pcc' / 'allen_visual_behavior_neuropixels_local.yaml'
checkpoint_options = sorted(str(path) for path in checkpoint_dir.glob('*_best.pt'))
checkpoint_widget = widgets.Dropdown(options=checkpoint_options, description='Checkpoint:')
display(checkpoint_widget)
assert checkpoint_options, 'No best checkpoints found. Run the training notebook first.'

In [ ]:
selected_checkpoint = checkpoint_widget.value
paths_ok = verify_paths_exist({
    'experiment_config': experiment_config,
    'data_config': data_config,
    'checkpoint': selected_checkpoint,
})
print(paths_ok)
print('Subset selection is controlled by dataset_selection in the experiment config when active.')
assert all(paths_ok.values()), 'Missing discovery/validation inputs.'

In [ ]:
reporter.begin('discovery', next_artifact='discovery artifact + cluster summary json/csv')
%cd {repo_root}
!pcc-discover --config {experiment_config} --data-config {data_config} --checkpoint {selected_checkpoint} --split discovery
reporter.finish('discovery')

In [ ]:
discovery_artifact = Path(selected_checkpoint).with_name(f"{Path(selected_checkpoint).stem}_discovery_discovery.json")
cluster_summary_json = discovery_artifact.with_name(f"{discovery_artifact.stem}_cluster_summary.json")
cluster_summary_csv = discovery_artifact.with_name(f"{discovery_artifact.stem}_cluster_summary.csv")
reporter.begin('validation', next_artifact='validation summary json/csv')
%cd {repo_root}
!pcc-validate --config {experiment_config} --data-config {data_config} --checkpoint {selected_checkpoint} --discovery-artifact {discovery_artifact}
reporter.finish('validation')

In [ ]:
validation_json = discovery_artifact.with_name(f"{discovery_artifact.stem}_validation.json")
validation_csv = discovery_artifact.with_name(f"{discovery_artifact.stem}_validation.csv")
with open(discovery_artifact, 'r', encoding='utf-8') as handle:
    discovery_payload = json.load(handle)
with open(cluster_summary_json, 'r', encoding='utf-8') as handle:
    cluster_summary_payload = json.load(handle)
with open(validation_json, 'r', encoding='utf-8') as handle:
    validation_payload = json.load(handle)
display(pd.read_csv(cluster_summary_csv))
display(pd.DataFrame(discovery_payload['candidates']).head())
display(pd.read_csv(validation_csv))
print('Cluster count:', cluster_summary_payload['cluster_count'])
print('Recurrence summary:', validation_payload['recurrence_summary'])